In [2]:
!pip install imbalanced-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectFromModel

from imblearn.over_sampling import SMOTE
import xgboost as xgb

# ================================
# LOAD DATA
# ================================
data = pd.read_csv("Data.csv")
labels = pd.read_csv("Lable.csv")

data['label'] = labels.iloc[:, -1]
df = data.copy()

# ================================
# CLEANING
# ================================
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

df['label'] = df['label'].apply(lambda x: 0 if x == 0 else 1)

# ================================
# PREPROCESS
# ================================
X = df.drop("label", axis=1)
y = df["label"]

X = X.select_dtypes(include=[np.number])

scaler = RobustScaler()
X = scaler.fit_transform(X)

# ================================
# SPLIT
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ================================
# SMOTE (OPTIMIZED)
# ================================
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# ================================
# FEATURE SELECTION
# ================================
selector = SelectFromModel(RandomForestClassifier(n_estimators=100, n_jobs=-1))
X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)

# ================================
# METRICS FUNCTION
# ================================
def evaluate(y_true, y_pred, name):
    print(f"\n===== {name} =====")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1 Score :", f1_score(y_true, y_pred))

# ============================================================
# RANDOM FOREST
# ============================================================
rf = RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1)
rf.fit(X_train, y_train)
evaluate(y_test, rf.predict(X_test), "RANDOM FOREST")

# ============================================================
# XGBOOST
# ============================================================
xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
evaluate(y_test, xgb_model.predict(X_test), "XGBOOST")

# ============================================================
# KMEANS + XGBOOST
# ============================================================
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
kmeans.fit(X_train)

X_train_km = np.column_stack((X_train, kmeans.predict(X_train)))
X_test_km = np.column_stack((X_test, kmeans.predict(X_test)))

xgb_km = xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.05)
xgb_km.fit(X_train_km, y_train)
evaluate(y_test, xgb_km.predict(X_test_km), "KMEANS + XGBOOST")

# ============================================================
# ISOLATION FOREST + XGBOOST
# ============================================================
iso = IsolationForest(contamination=0.03, n_estimators=150, random_state=42)
iso.fit(X_train)

X_train_iso = np.column_stack((X_train, iso.predict(X_train)))
X_test_iso = np.column_stack((X_test, iso.predict(X_test)))

xgb_iso = xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.05)
xgb_iso.fit(X_train_iso, y_train)
evaluate(y_test, xgb_iso.predict(X_test_iso), "ISOLATION + XGBOOST")

# ============================================================
# PCA + XGBOOST
# ============================================================
pca = PCA(n_components=20)
X_pca = pca.fit_transform(X)

X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

xgb_pca = xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.05)
xgb_pca.fit(X_train_pca, y_train_pca)
evaluate(y_test_pca, xgb_pca.predict(X_test_pca), "PCA + XGBOOST")

# ============================================================
# 💀 ULTIMATE HYBRID (FAST + STABLE)
# ============================================================
km_feat_train = kmeans.predict(X_train)
iso_feat_train = iso.predict(X_train)

km_feat_test = kmeans.predict(X_test)
iso_feat_test = iso.predict(X_test)

X_train_hybrid = np.column_stack((X_train, km_feat_train, iso_feat_train))
X_test_hybrid = np.column_stack((X_test, km_feat_test, iso_feat_test))

xgb_hybrid = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=10,
    learning_rate=0.05,
    eval_metric='logloss'
)

xgb_hybrid.fit(X_train_hybrid, y_train)
evaluate(y_test, xgb_hybrid.predict(X_test_hybrid), "💀 ULTIMATE HYBRID")


===== RANDOM FOREST =====
Accuracy : 0.9825524932185794
Precision: 0.9246468633153303
Recall   : 0.9937489535078418
F1 Score : 0.957953353240255

===== XGBOOST =====
Accuracy : 0.9827534241987878
Precision: 0.9249818295088775
Recall   : 0.9944187084891444
F1 Score : 0.9584442830630194

===== KMEANS + XGBOOST =====
Accuracy : 0.9822287710837994
Precision: 0.9219436546911347
Recall   : 0.9954233409610984
F1 Score : 0.9572755085610005

===== ISOLATION + XGBOOST =====
Accuracy : 0.9822287710837994
Precision: 0.9219436546911347
Recall   : 0.9954233409610984
F1 Score : 0.9572755085610005

===== PCA + XGBOOST =====
Accuracy : 0.9797729479923646
Precision: 0.9167744940738057
Recall   : 0.9886141653178545
F1 Score : 0.9513400290026317

===== 💀 ULTIMATE HYBRID =====
Accuracy : 0.9825859817152808
Precision: 0.9243941674017955
Recall   : 0.9942512697438187
F1 Score : 0.9580509841884479
